# Lab 007 — Make the leak, then kill it

**Lesson:** [`lessons/0007-class-imbalance.html`](../lessons/0007-class-imbalance.html) · **Phase / Year:** Phase 1 / Year 1

**Skill you are practising:** on imbalanced data, distrust accuracy, judge with imbalance-aware metrics, and resample only *inside* the `imblearn` Pipeline.

**Exit criteria:** the EXIT TICKET prints the accuracy paradox (high accuracy, ~0 recall), the SMOTE leaked-vs-honest F1, and the `class_weight` recall gain, plus your one-sentence cause.

---

### How this notebook works
- **PROVIDED** cells are complete — just run them.
- **TODO** cells have blanks (`____`). Fill them in.
- **CHECK** cells auto-run and give you immediate feedback — don't edit them.
- Run top to bottom. When the **EXIT TICKET** prints cleanly, paste it back to your teacher (or say *"lab done"*).

### Environment
One-time setup from the repo root: `bash labs/setup-env.sh` (installs `imbalanced-learn`).  
Then select kernel **Relational Labs (.venv)** or interpreter `.venv/bin/python`.

## Setup — PROVIDED (run me)

In [ ]:
# PROVIDED
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, average_precision_score
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from imblearn.over_sampling import SMOTE

RS = 0
print("setup ok")

## Data — PROVIDED

A binary dataset that is about **5% positive** — the rare class is what we care about.

In [ ]:
# PROVIDED
X, y = make_classification(n_samples=5000, n_features=20, n_informative=5,
                           weights=[0.95, 0.05], random_state=RS)
cv = StratifiedKFold(5, shuffle=True, random_state=RS)
print("positive rate:", round(y.mean(), 4))

## Task 1 — The accuracy paradox — TODO

Fit a model that **always predicts the majority class** and score it. Use
`DummyClassifier(strategy="most_frequent")`. Watch accuracy stay high while recall is zero.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RS)

# TODO: fit the majority-class dummy and predict on the test set.
dummy = ____
dummy_pred = ____

dummy_acc = accuracy_score(yte, dummy_pred)
dummy_recall = recall_score(yte, dummy_pred, zero_division=0)
print(f"accuracy {dummy_acc:.3f}   recall {dummy_recall:.3f}")

In [ ]:
# CHECK — do not edit.
assert dummy_acc > 0.90, "Majority-class accuracy should be high (>0.90)."
assert dummy_recall == 0.0, "A majority-class predictor catches zero positives — recall must be 0."
print("Task 1 ok — high accuracy, zero recall: the accuracy paradox.")

## Task 2 — The SMOTE leak — TODO

Compare two protocols by 5-fold **F1**:
- **WRONG:** SMOTE the *entire* dataset, then `cross_val_score` on the resampled data.
- **RIGHT:** put SMOTE in an `imblearn` Pipeline so it only touches training folds.

Hint: `X_res, y_res = SMOTE(random_state=RS).fit_resample(X, y)` for the wrong one;
`imb_make_pipeline(SMOTE(random_state=RS), LogisticRegression(max_iter=2000))` for the right one.

In [ ]:
clf = LogisticRegression(max_iter=2000)

# TODO (WRONG): resample everything, then cross-validate on it.
X_res, y_res = ____
leaked_f1 = cross_val_score(clf, X_res, y_res, cv=cv, scoring="f1").mean()

# TODO (RIGHT): SMOTE inside an imblearn pipeline.
honest_pipe = ____
honest_f1 = cross_val_score(honest_pipe, X, y, cv=cv, scoring="f1").mean()

print(f"leaked F1 (resample-all then CV): {leaked_f1:.3f}")
print(f"honest F1 (SMOTE in pipeline)   : {honest_f1:.3f}")
print(f"fiction manufactured            : {leaked_f1 - honest_f1:+.3f}")

In [ ]:
# CHECK — do not edit.
assert leaked_f1 > honest_f1 + 0.10, "The leaked F1 should be much higher than the honest one."
print(f"Task 2 ok — resampling before CV inflated F1 by {leaked_f1 - honest_f1:.3f} of fiction.")

## Task 3 — The leak-free fix: class weights — TODO

No resampling at all. Compare 5-fold **recall** of a plain logistic regression vs one with
`class_weight="balanced"`. Also look at PR-AUC (`scoring="average_precision"`) to see that
reweighting shifts the operating point more than it adds ranking signal.

In [ ]:
# TODO: fill the class_weight argument for the balanced model.
plain_recall    = cross_val_score(LogisticRegression(max_iter=2000),
                                  X, y, cv=cv, scoring="recall").mean()
weighted_recall = cross_val_score(LogisticRegression(max_iter=2000, class_weight=____),
                                  X, y, cv=cv, scoring="recall").mean()

plain_ap    = cross_val_score(LogisticRegression(max_iter=2000),
                              X, y, cv=cv, scoring="average_precision").mean()
weighted_ap = cross_val_score(LogisticRegression(max_iter=2000, class_weight="balanced"),
                              X, y, cv=cv, scoring="average_precision").mean()

print(f"recall  plain {plain_recall:.3f} -> balanced {weighted_recall:.3f}")
print(f"PR-AUC  plain {plain_ap:.3f} -> balanced {weighted_ap:.3f}")

In [ ]:
# CHECK — do not edit.
assert weighted_recall > plain_recall + 0.10, "Balanced weights should raise recall noticeably."
print(f"Task 3 ok — class_weight raised recall by {weighted_recall - plain_recall:.3f} "
      f"(PR-AUC moved {weighted_ap - plain_ap:+.3f}: it shifts the operating point, not the ranking).")

## Exit ticket — TODO

Fill the one-sentence takeaway, run, and paste the output back to your teacher.

In [ ]:
# TODO: complete the takeaway.
print("=== EXIT TICKET — Lesson 007 ===")
print(f"accuracy paradox : acc {dummy_acc:.3f}, recall {dummy_recall:.3f}")
print(f"SMOTE F1         : leaked {leaked_f1:.3f} vs honest {honest_f1:.3f}")
print(f"class_weight     : recall {plain_recall:.3f} -> {weighted_recall:.3f}")
print()
print("takeaway:", "____")   # why was the leaked F1 higher, and which fix is leak-free?